In [1]:
# Cell 1 — Install dependencies
!pip install -q "transformers>=4.46,<5" datasets accelerate sentencepiece scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 143.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Cell 3 — Imports
import os
import re
import gc
import glob
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    set_seed,
)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)

In [4]:
# Cell 4 — Master experiment configuration
# If your Drive folder has a different full name, change BASE_DIR only.

BASE_DIR = '/content/drive/MyDrive/5cv_experiement_intent_11Aug2026'


# Optional auto-detection if the exact folder name is longer than the screenshot shows.
if not os.path.isdir(BASE_DIR):
    candidates = sorted(glob.glob('/content/drive/MyDrive/5cv_experiement_intent_11Aug2026*'))
    if len(candidates) == 1:
        BASE_DIR = candidates[0]
        print('Auto-detected BASE_DIR:', BASE_DIR)
    else:
        raise FileNotFoundError(
            'Set BASE_DIR to your 5-CV experiment folder. Candidates found: '
            + str(candidates)
        )

SOURCE_PATH = os.path.join(BASE_DIR, 'source', 'conversation_en.json')
TRANSLATED_PATH = os.path.join(
    BASE_DIR,
    'translated_data',
    'conversation_cn_translated_en_deepseek_v4flash.json',
)
SPLIT_DIR = os.path.join(BASE_DIR, 'splits')
LLM_AUG_DIR = os.path.join(BASE_DIR, 'llm_augmentation')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

VALID_LABELS = ['Chitchat', 'Interpret', 'Reveal', 'Revise']
label2id = {label: i for i, label in enumerate(VALID_LABELS)}
id2label = {i: label for label, i in label2id.items()}

RANDOM_STATE = 42
MAX_LEN = 256
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8

# SAME optimization hyperparameters for all four models.
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

RUN_FOLDS = [1, 2, 3, 4, 5]

MODEL_CONFIGS = {
    'BERT-base': {
        'model_name': 'bert-base-uncased',
        'enabled': False,
    },
    'DistilBERT': {
        'model_name': 'distilbert-base-uncased',
        'enabled': False,
    },
    'RoBERTa': {
        'model_name': 'roberta-base',
        'enabled': True,
    },
    'MiniLM': {
        'model_name': 'microsoft/MiniLM-L12-H384-uncased',
        'enabled': True,
    },
}

print('BASE_DIR:', BASE_DIR)
print('RUN_FOLDS:', RUN_FOLDS)
print('Enabled models:', [k for k,v in MODEL_CONFIGS.items() if v['enabled']])
print('Shared LR:', LEARNING_RATE, '| Shared weight decay:', WEIGHT_DECAY)

BASE_DIR: /content/drive/MyDrive/5cv_experiement_intent_11Aug2026
RUN_FOLDS: [1, 2, 3, 4, 5]
Enabled models: ['RoBERTa', 'MiniLM']
Shared LR: 2e-05 | Shared weight decay: 0.01


In [5]:
# Cell 5 — Verify expected files exist
required_paths = [SOURCE_PATH, TRANSLATED_PATH]
for cv in range(1, 6):
    required_paths.extend([
        os.path.join(SPLIT_DIR, f'cv{cv}_train_ids.csv'),
        os.path.join(SPLIT_DIR, f'cv{cv}_val_ids.csv'),
        os.path.join(SPLIT_DIR, f'cv{cv}_test_ids.csv'),
    ])

missing = [p for p in required_paths if not os.path.exists(p)]
assert not missing, 'Missing required files:\n' + '\n'.join(missing)

print('PASS: source, translated corpus, and all 15 frozen CV split files exist.')

PASS: source, translated corpus, and all 15 frozen CV split files exist.


## Data loading and context construction

The original English corpus keeps **all dialogue turns** for history construction, while only the four target intents are prediction targets. `Inquire` may therefore appear in history but is not a target label.

The translated corpus is namespaced with `CN_` conversation IDs so coincident numeric IDs can never overwrite or collide with English conversations.

In [6]:
# Cell 6 — Generic conversation JSON parser

def conversations_to_turn_df(raw_conversations, id_prefix=''):
    rows = []
    for conv in raw_conversations:
        cid = id_prefix + str(conv['conv_id'])
        for turn_index, turn in enumerate(conv.get('conversation', [])):
            rows.append({
                'conversation_id': cid,
                'turn_index': int(turn_index),
                'role': str(turn.get('role', '')).lower().strip(),
                'text': turn.get('content', '') or '',
                'intent': turn.get('intent'),
            })
    return (
        pd.DataFrame(rows)
        .sort_values(['conversation_id', 'turn_index'])
        .reset_index(drop=True)
    )


def build_turn_lookup(turn_df):
    return {
        str(cid): g.sort_values('turn_index').reset_index(drop=True)
        for cid, g in turn_df.groupby('conversation_id')
    }


def build_context_dataset(target_rows, turns_by_conv, return_meta=False):
    texts, labels, meta = [], [], []

    for _, row in target_rows.iterrows():
        cid = str(row['conversation_id'])
        ti = int(row['turn_index'])
        assert cid in turns_by_conv, f'Missing history for conversation {cid}'

        conv_turns = turns_by_conv[cid]
        previous = conv_turns[conv_turns['turn_index'] < ti]

        history = []
        for _, p in previous.iterrows():
            txt = str(p['text']).strip()
            if not txt:
                continue
            history.append(f"{str(p['role']).upper()}: {txt}")

        current = str(row['text']).strip()
        full_text = ('\n'.join(history) + '\nUSER: ' + current) if history else ('USER: ' + current)

        texts.append(full_text)
        labels.append(str(row['intent']))
        meta.append((cid, ti))

    if return_meta:
        return np.array(texts), np.array(labels), meta
    return np.array(texts), np.array(labels)

In [7]:
# Cell 7 — Load original English corpus
with open(SOURCE_PATH, 'r', encoding='utf-8') as f:
    raw_en = json.load(f)

en_turns_full = conversations_to_turn_df(raw_en)
en_turns_by_conv = build_turn_lookup(en_turns_full)

en_target_df = en_turns_full[
    (en_turns_full['role'] == 'user')
    & (en_turns_full['intent'].isin(VALID_LABELS))
].copy().reset_index(drop=True)

print('Original English conversations:', en_turns_full['conversation_id'].nunique())
print('4-class eligible English conversations:', en_target_df['conversation_id'].nunique())
print('4-class English target turns:', len(en_target_df))
print('\nEnglish target distribution:')
print(en_target_df['intent'].value_counts().reindex(VALID_LABELS, fill_value=0))

assert en_target_df['conversation_id'].nunique() == 825
assert len(en_target_df) == 3179

Original English conversations: 826
4-class eligible English conversations: 825
4-class English target turns: 3179

English target distribution:
intent
Chitchat      268
Interpret    1813
Reveal       1031
Revise         67
Name: count, dtype: int64


In [8]:
# Cell 8 — Load independent translated Chinese→English corpus, namespace IDs
with open(TRANSLATED_PATH, 'r', encoding='utf-8') as f:
    raw_cn = json.load(f)

# Capture raw numeric/string IDs only for diagnostics; they are NOT used for joining.
raw_cn_ids = {str(conv['conv_id']) for conv in raw_cn}
raw_en_ids = {str(conv['conv_id']) for conv in raw_en}
raw_numeric_id_overlap = raw_cn_ids & raw_en_ids
print('Raw English/CN numeric ID overlap before namespacing:', len(raw_numeric_id_overlap))
print('This is not treated as leakage because the translated corpus is independent; CN IDs are namespaced below.')

cn_turns_full = conversations_to_turn_df(raw_cn, id_prefix='CN_')
cn_turns_by_conv = build_turn_lookup(cn_turns_full)
cn_target_df = cn_turns_full[
    (cn_turns_full['role'] == 'user')
    & (cn_turns_full['intent'].isin(VALID_LABELS))
].copy().reset_index(drop=True)

assert cn_turns_full['conversation_id'].astype(str).str.startswith('CN_').all()
assert cn_target_df['conversation_id'].astype(str).str.startswith('CN_').all()

X_cn_train, y_cn_train, cn_meta = build_context_dataset(
    cn_target_df, cn_turns_by_conv, return_meta=True
)

print('\nTranslated CN→EN target turns:', len(cn_target_df))
print('Translated label distribution:')
print(cn_target_df['intent'].value_counts().reindex(VALID_LABELS, fill_value=0))

Raw English/CN numeric ID overlap before namespacing: 6
This is not treated as leakage because the translated corpus is independent; CN IDs are namespaced below.

Translated CN→EN target turns: 3055
Translated label distribution:
intent
Chitchat      275
Interpret    1590
Reveal        975
Revise        215
Name: count, dtype: int64


## Frozen split and augmentation loaders

No random splitting occurs below. All membership comes from the frozen CSV files.

In [9]:
# Cell 9 — Load one frozen CV split

def load_cv_ids(cv_run):
    train_ids = set(pd.read_csv(os.path.join(SPLIT_DIR, f'cv{cv_run}_train_ids.csv'))['conv_id'].astype(str))
    val_ids = set(pd.read_csv(os.path.join(SPLIT_DIR, f'cv{cv_run}_val_ids.csv'))['conv_id'].astype(str))
    test_ids = set(pd.read_csv(os.path.join(SPLIT_DIR, f'cv{cv_run}_test_ids.csv'))['conv_id'].astype(str))
    return train_ids, val_ids, test_ids


def find_llm_aug_file(cv_run):
    matches = sorted(glob.glob(os.path.join(
        LLM_AUG_DIR,
        f'cv{cv_run}_synthetic_100_chitchat_100_revise*.csv'
    )))
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Expected exactly one CV{cv_run} LLM augmentation file; found {len(matches)}: {matches}'
        )
    return matches[0]


def load_llm_augmentation(cv_run):
    path = find_llm_aug_file(cv_run)
    df = pd.read_csv(path)
    df['conversation_id'] = df['conversation_id'].astype(str)
    df['turn_index'] = df['turn_index'].astype(int)
    df['intent'] = df['intent'].astype(str).str.strip()
    df['text'] = df['text'].fillna('').astype(str)
    return df, path

In [10]:
# Cell 10 — Deep leakage audit for a single CV fold

def audit_cv_fold(cv_run, verbose=True):
    train_ids, val_ids, test_ids = load_cv_ids(cv_run)
    aug_df, aug_path = load_llm_augmentation(cv_run)

    # 1) Frozen conversation memberships
    assert len(train_ids) == 528, f'CV{cv_run}: train conversation count != 528'
    assert len(val_ids) == 132, f'CV{cv_run}: val conversation count != 132'
    assert len(test_ids) == 165, f'CV{cv_run}: test conversation count != 165'
    assert train_ids.isdisjoint(val_ids), f'CV{cv_run}: train/val conversation leakage'
    assert train_ids.isdisjoint(test_ids), f'CV{cv_run}: train/test conversation leakage'
    assert val_ids.isdisjoint(test_ids), f'CV{cv_run}: val/test conversation leakage'
    assert len(train_ids | val_ids | test_ids) == 825, f'CV{cv_run}: split union != 825'

    # 2) All IDs must come from the 825 eligible English conversations
    eligible_ids = set(en_target_df['conversation_id'].astype(str))
    assert (train_ids | val_ids | test_ids) == eligible_ids, f'CV{cv_run}: split IDs do not exactly match eligible English IDs'

    # 3) Optional outer-fold file must match corresponding CV test IDs if present
    fold_path = os.path.join(SPLIT_DIR, f'fold_{cv_run}_ids.csv')
    if os.path.exists(fold_path):
        fold_ids = set(pd.read_csv(fold_path)['conv_id'].astype(str))
        assert fold_ids == test_ids, f'CV{cv_run}: cv test IDs do not equal fold_{cv_run}_ids.csv'

    # 4) Reconstruct English target rows and verify no target-key overlap
    train_df = en_target_df[en_target_df['conversation_id'].isin(train_ids)].copy()
    val_df = en_target_df[en_target_df['conversation_id'].isin(val_ids)].copy()
    test_df = en_target_df[en_target_df['conversation_id'].isin(test_ids)].copy()

    key = lambda d: set(zip(d['conversation_id'].astype(str), d['turn_index'].astype(int)))
    train_keys, val_keys, test_keys = key(train_df), key(val_df), key(test_df)
    assert train_keys.isdisjoint(val_keys), f'CV{cv_run}: target-turn train/val leakage'
    assert train_keys.isdisjoint(test_keys), f'CV{cv_run}: target-turn train/test leakage'
    assert val_keys.isdisjoint(test_keys), f'CV{cv_run}: target-turn val/test leakage'

    # 5) LLM augmentation provenance
    required_aug_cols = {
        'conversation_id', 'turn_index', 'role', 'text', 'intent',
        'is_augmented', 'source_user_text'
    }
    missing_cols = required_aug_cols - set(aug_df.columns)
    assert not missing_cols, f'CV{cv_run}: missing LLM columns: {missing_cols}'
    assert len(aug_df) == 200, f'CV{cv_run}: LLM synthetic rows != 200'

    aug_ids = set(aug_df['conversation_id'].astype(str))
    assert aug_ids.issubset(train_ids), f'CV{cv_run}: LLM source outside frozen train'
    assert aug_ids.isdisjoint(val_ids), f'CV{cv_run}: LLM source overlaps validation'
    assert aug_ids.isdisjoint(test_ids), f'CV{cv_run}: LLM source overlaps test'

    aug_counts = aug_df['intent'].value_counts()
    assert aug_counts.get('Chitchat', 0) == 100, f'CV{cv_run}: Chitchat synthetic count != 100'
    assert aug_counts.get('Revise', 0) == 100, f'CV{cv_run}: Revise synthetic count != 100'
    assert set(aug_df['intent'].dropna()) <= {'Chitchat', 'Revise'}

    if 'is_augmented' in aug_df.columns:
        aug_flag = aug_df['is_augmented'].astype(str).str.lower().isin(['true','1','1.0'])
        assert aug_flag.all(), f'CV{cv_run}: non-augmented row found in synthetic file'

    dup_syn = aug_df.duplicated(['conversation_id','turn_index','text','intent']).sum()
    assert dup_syn == 0, f'CV{cv_run}: duplicate synthetic rows found'

    # 6) Every synthetic row must map to a genuine original TRAIN target turn with SAME intent.
    original_train_map = {
        (str(r.conversation_id), int(r.turn_index)): str(r.intent)
        for r in train_df[['conversation_id','turn_index','intent']].itertuples(index=False)
    }
    bad_source = []
    bad_label = []
    for r in aug_df[['conversation_id','turn_index','intent']].itertuples(index=False):
        k = (str(r.conversation_id), int(r.turn_index))
        if k not in original_train_map:
            bad_source.append(k)
        elif original_train_map[k] != str(r.intent):
            bad_label.append((k, original_train_map[k], str(r.intent)))
    assert not bad_source, f'CV{cv_run}: synthetic rows with missing original train source turn: {bad_source[:5]}'
    assert not bad_label, f'CV{cv_run}: synthetic labels differ from original source label: {bad_label[:5]}'

    # 7) Independent translated corpus is structurally isolated by namespace.
    assert cn_target_df['conversation_id'].astype(str).str.startswith('CN_').all()
    assert not any(str(x).startswith('CN_') for x in train_ids | val_ids | test_ids)

    # Distribution summaries
    def counts_dict(df):
        c = df['intent'].value_counts().reindex(VALID_LABELS, fill_value=0)
        return {label: int(c[label]) for label in VALID_LABELS}

    train_counts = counts_dict(train_df)
    val_counts = counts_dict(val_df)
    test_counts = counts_dict(test_df)
    cn_counts = cn_target_df['intent'].value_counts().reindex(VALID_LABELS, fill_value=0)
    aug_by_label = aug_df['intent'].value_counts().reindex(VALID_LABELS, fill_value=0)

    final_train_counts = {
        label: train_counts[label] + int(aug_by_label[label]) + int(cn_counts[label])
        for label in VALID_LABELS
    }

    row = {
        'CV': cv_run,
        'Train_conversations': len(train_ids),
        'Val_conversations': len(val_ids),
        'Test_conversations': len(test_ids),
        'Train_original_targets': len(train_df),
        'Val_targets': len(val_df),
        'Test_targets': len(test_df),
        'LLM_rows': len(aug_df),
        'LLM_source_conversations': len(aug_ids),
        'Translated_targets': len(cn_target_df),
        'Final_train_examples': len(train_df) + len(aug_df) + len(cn_target_df),
    }
    for label in VALID_LABELS:
        row[f'TrainOriginal_{label}'] = train_counts[label]
        row[f'Val_{label}'] = val_counts[label]
        row[f'Test_{label}'] = test_counts[label]
        row[f'LLM_{label}'] = int(aug_by_label[label])
        row[f'Translated_{label}'] = int(cn_counts[label])
        row[f'FinalTrain_{label}'] = final_train_counts[label]

    if verbose:
        print(f'CV{cv_run}: PASS — no conversation-ID or target-turn leakage; LLM provenance verified.')
        print('  Train/Val/Test conversations:', len(train_ids), len(val_ids), len(test_ids))
        print('  Original target rows:', len(train_df), len(val_df), len(test_df))
        print('  LLM rows / source conversations:', len(aug_df), len(aug_ids))

    return row

In [11]:
# Cell 11 — Audit ALL 5 CV folds before any training
fold_audit_rows = [audit_cv_fold(cv, verbose=True) for cv in range(1, 6)]
fold_data_summary = pd.DataFrame(fold_audit_rows)

# Across outer CV: every eligible conversation must be TEST exactly once.
all_test_ids = []
for cv in range(1, 6):
    _, _, test_ids = load_cv_ids(cv)
    all_test_ids.extend(sorted(test_ids))

assert len(all_test_ids) == 825
assert len(set(all_test_ids)) == 825
assert set(all_test_ids) == set(en_target_df['conversation_id'].astype(str))

# Pairwise outer test folds must be disjoint.
for i in range(1, 6):
    ti = load_cv_ids(i)[2]
    for j in range(i + 1, 6):
        tj = load_cv_ids(j)[2]
        assert ti.isdisjoint(tj), f'Outer test fold overlap between CV{i} and CV{j}'

print('\nPASS: all five outer test folds are pairwise disjoint and cover all 825 eligible conversations exactly once.')

summary_path = os.path.join(RESULTS_DIR, 'fold_data_and_leakage_audit.csv')
fold_data_summary.to_csv(summary_path, index=False)
print('Saved fold audit summary:', summary_path)

display(fold_data_summary)

CV1: PASS — no conversation-ID or target-turn leakage; LLM provenance verified.
  Train/Val/Test conversations: 528 132 165
  Original target rows: 2035 489 655
  LLM rows / source conversations: 200 66
CV2: PASS — no conversation-ID or target-turn leakage; LLM provenance verified.
  Train/Val/Test conversations: 528 132 165
  Original target rows: 2053 503 623
  LLM rows / source conversations: 200 66
CV3: PASS — no conversation-ID or target-turn leakage; LLM provenance verified.
  Train/Val/Test conversations: 528 132 165
  Original target rows: 2041 504 634
  LLM rows / source conversations: 200 62
CV4: PASS — no conversation-ID or target-turn leakage; LLM provenance verified.
  Train/Val/Test conversations: 528 132 165
  Original target rows: 2038 507 634
  LLM rows / source conversations: 200 64
CV5: PASS — no conversation-ID or target-turn leakage; LLM provenance verified.
  Train/Val/Test conversations: 528 132 165
  Original target rows: 2020 526 633
  LLM rows / source convers

,CV,Train_conversations,Val_conversations,Test_conversations,Train_original_targets,Val_targets,Test_targets,LLM_rows,LLM_source_conversations,Translated_targets,Final_train_examples,TrainOriginal_Chitchat,Val_Chitchat,Test_Chitchat,LLM_Chitchat,Translated_Chitchat,FinalTrain_Chitchat,TrainOriginal_Interpret,Val_Interpret,Test_Interpret,LLM_Interpret,Translated_Interpret,FinalTrain_Interpret,TrainOriginal_Reveal,Val_Reveal,Test_Reveal,LLM_Reveal,Translated_Reveal,FinalTrain_Reveal,TrainOriginal_Revise,Val_Revise,Test_Revise,LLM_Revise,Translated_Revise,FinalTrain_Revise
0,1,528,132,165,2035,489,655,200,66,3055,5290,176,39,53,100,275,551,1143,293,377,0,1590,2733,671,150,210,0,975,1646,45,7,15,100,215,360
1,2,528,132,165,2053,503,623,200,66,3055,5308,179,39,50,100,275,554,1158,297,358,0,1590,2748,670,160,201,0,975,1645,46,7,14,100,215,361
2,3,528,132,165,2041,504,634,200,62,3055,5296,179,37,52,100,275,554,1163,299,351,0,1590,2753,653,161,217,0,975,1628,46,7,14,100,215,361
3,4,528,132,165,2038,507,634,200,64,3055,5293,179,36,53,100,275,554,1146,299,368,0,1590,2736,665,165,201,0,975,1640,48,7,12,100,215,363
4,5,528,132,165,2020,526,633,200,62,3055,5275,172,36,60,100,275,547,1136,318,359,0,1590,2726,664,165,202,0,975,1639,48,7,12,100,215,363


In [12]:
# Cell 12 — Compact original split distribution table for all five CV runs
rows = []
for cv in range(1, 6):
    train_ids, val_ids, test_ids = load_cv_ids(cv)
    for split_name, ids in [('Train', train_ids), ('Validation', val_ids), ('Test', test_ids)]:
        d = en_target_df[en_target_df['conversation_id'].isin(ids)]
        c = d['intent'].value_counts().reindex(VALID_LABELS, fill_value=0)
        rows.append({
            'CV': f'CV{cv}',
            'Split': split_name,
            'Conversations': len(ids),
            'Chitchat': int(c['Chitchat']),
            'Interpret': int(c['Interpret']),
            'Reveal': int(c['Reveal']),
            'Revise': int(c['Revise']),
            'Total_examples': int(len(d)),
        })

original_split_distribution = pd.DataFrame(rows)
display(original_split_distribution)
original_split_distribution.to_csv(
    os.path.join(RESULTS_DIR, 'original_cv_split_distribution.csv'),
    index=False,
)

,CV,Split,Conversations,Chitchat,Interpret,Reveal,Revise,Total_examples
0,CV1,Train,528,176,1143,671,45,2035
1,CV1,Validation,132,39,293,150,7,489
2,CV1,Test,165,53,377,210,15,655
3,CV2,Train,528,179,1158,670,46,2053
4,CV2,Validation,132,39,297,160,7,503
5,CV2,Test,165,50,358,201,14,623
6,CV3,Train,528,179,1163,653,46,2041
7,CV3,Validation,132,37,299,161,7,504
8,CV3,Test,165,52,351,217,14,634
9,CV4,Train,528,179,1146,665,48,2038


## Model-ready fold construction

This stage performs the same leakage checks again immediately before each training run, builds original-English contexts, injects only the matching CV-specific LLM rows, and appends the independent translated corpus to training only.

In [13]:
# Cell 13 — Build model-ready data for one CV run

def build_one_cv_data(cv_run):
    # Repeat deep audit immediately before data assembly.
    audit_cv_fold(cv_run, verbose=False)

    train_ids, val_ids, test_ids = load_cv_ids(cv_run)
    aug_df, aug_path = load_llm_augmentation(cv_run)

    train_original_df = en_target_df[en_target_df['conversation_id'].isin(train_ids)].copy().reset_index(drop=True)
    val_df = en_target_df[en_target_df['conversation_id'].isin(val_ids)].copy().reset_index(drop=True)
    test_df = en_target_df[en_target_df['conversation_id'].isin(test_ids)].copy().reset_index(drop=True)

    X_train_original, y_train_original, train_meta = build_context_dataset(
        train_original_df, en_turns_by_conv, return_meta=True
    )
    X_val, y_val, val_meta = build_context_dataset(
        val_df, en_turns_by_conv, return_meta=True
    )
    X_test, y_test, test_meta = build_context_dataset(
        test_df, en_turns_by_conv, return_meta=True
    )
    X_llm, y_llm, llm_meta = build_context_dataset(
        aug_df, en_turns_by_conv, return_meta=True
    )

    # Provenance-level key overlap checks (synthetic intentionally shares source TRAIN keys).
    train_conv = {cid for cid, _ in train_meta}
    val_conv = {cid for cid, _ in val_meta}
    test_conv = {cid for cid, _ in test_meta}
    llm_conv = {cid for cid, _ in llm_meta}
    assert train_conv.isdisjoint(val_conv)
    assert train_conv.isdisjoint(test_conv)
    assert val_conv.isdisjoint(test_conv)
    assert llm_conv.issubset(train_conv)
    assert llm_conv.isdisjoint(val_conv)
    assert llm_conv.isdisjoint(test_conv)

    # Translated data is training-only and namespaced.
    assert all(str(cid).startswith('CN_') for cid, _ in cn_meta)
    assert not any(str(cid).startswith('CN_') for cid, _ in val_meta + test_meta)

    X_train = np.concatenate([X_train_original, X_llm, X_cn_train])
    y_train = np.concatenate([y_train_original, y_llm, y_cn_train])

    return {
        'X_train': X_train,
        'y_train': y_train,
        'X_val': X_val,
        'y_val': y_val,
        'X_test': X_test,
        'y_test': y_test,
        'train_original_df': train_original_df,
        'val_df': val_df,
        'test_df': test_df,
        'aug_df': aug_df,
        'train_ids': train_ids,
        'val_ids': val_ids,
        'test_ids': test_ids,
        'test_meta': test_meta,
        'aug_path': aug_path,
    }

In [14]:
# Cell 14 — Smoke-inspect CV1 model-ready distributions before training
smoke_data = build_one_cv_data(1)

print('\nCV1 final model-ready sizes:')
print('Original English train:', len(smoke_data['train_original_df']))
print('LLM synthetic:', len(smoke_data['aug_df']))
print('Translated CN→EN:', len(cn_target_df))
print('FINAL train:', len(smoke_data['X_train']))
print('Validation:', len(smoke_data['X_val']))
print('Test:', len(smoke_data['X_test']))

print('\nCV1 FINAL TRAIN distribution:')
print(pd.Series(smoke_data['y_train']).value_counts().reindex(VALID_LABELS, fill_value=0))
print('\nCV1 VAL distribution:')
print(pd.Series(smoke_data['y_val']).value_counts().reindex(VALID_LABELS, fill_value=0))
print('\nCV1 TEST distribution:')
print(pd.Series(smoke_data['y_test']).value_counts().reindex(VALID_LABELS, fill_value=0))


CV1 final model-ready sizes:
Original English train: 2035
LLM synthetic: 200
Translated CN→EN: 3055
FINAL train: 5290
Validation: 489
Test: 655

CV1 FINAL TRAIN distribution:
Chitchat      551
Interpret    2733
Reveal       1646
Revise        360
Name: count, dtype: int64

CV1 VAL distribution:
Chitchat      39
Interpret    293
Reveal       150
Revise         7
Name: count, dtype: int64

CV1 TEST distribution:
Chitchat      53
Interpret    377
Reveal       210
Revise        15
Name: count, dtype: int64


## Training utilities

All four models use the **same** learning rate (`2e-5`) and weight decay (`0.01`). Balanced class weights are computed from each fold's **final training data only**. Validation Macro-F1 selects the best in-memory model state; test data is never used for checkpoint selection.

In [15]:
# Cell 15 — Label encoding, datasets, metrics

def labels_to_ids(labels):
    return np.array([label2id[str(x)] for x in labels], dtype=np.int64)


def make_hf_datasets(tokenizer, data):
    def encode(texts):
        return tokenizer(
            list(texts),
            truncation=True,
            max_length=MAX_LEN,
        )

    y_train_ids = labels_to_ids(data['y_train'])
    y_val_ids = labels_to_ids(data['y_val'])
    y_test_ids = labels_to_ids(data['y_test'])

    train_ds = Dataset.from_dict({**encode(data['X_train']), 'labels': y_train_ids})
    val_ds = Dataset.from_dict({**encode(data['X_val']), 'labels': y_val_ids})
    test_ds = Dataset.from_dict({**encode(data['X_test']), 'labels': y_test_ids})

    return train_ds, val_ds, test_ds, y_train_ids, y_val_ids, y_test_ids


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        labels, preds,  labels=np.arange(len(VALID_LABELS)),average='macro', zero_division=0
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    _, _, class_f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        labels=np.arange(len(VALID_LABELS)),
        average=None,
        zero_division=0,
    )

    out = {
        'accuracy': accuracy_score(labels, preds),
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
    }
    for i, label in id2label.items():
        out[f'{label.lower()}_f1'] = float(class_f1[i])
    return out

In [16]:
# Cell 16 — Weighted Trainer and validation-only best-state callback

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits

        weight = self.class_weights.to(
        device=logits.device,
        dtype=logits.dtype,
        )

        loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss


class BestValidationStateCallback(TrainerCallback):
    def __init__(self, metric_name='eval_macro_f1'):
        self.metric_name = metric_name
        self.best_metric = -np.inf
        self.best_epoch = None
        self.best_step = None
        self.best_state_dict = None
        self.best_metrics = None

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if metrics is None or model is None:
            return control
        current = metrics.get(self.metric_name)
        if current is None:
            return control
        if float(current) > self.best_metric:
            self.best_metric = float(current)
            self.best_epoch = float(state.epoch) if state.epoch is not None else None
            self.best_step = int(state.global_step)
            self.best_state_dict = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
            self.best_metrics = {
                k: float(v)
                for k, v in metrics.items()
                if isinstance(v, (int, float, np.integer, np.floating))
            }
        return control

In [17]:
# Cell 17 — Test evaluator

def evaluate_test(trainer, test_ds, y_test_ids, data, model_label, cv_run, output_dir):
    pred = trainer.predict(test_ds)
    y_pred_ids = np.argmax(pred.predictions, axis=1)

    y_true = [id2label[int(i)] for i in y_test_ids]
    y_pred = [id2label[int(i)] for i in y_pred_ids]

    print(f'\n=== {model_label} | CV{cv_run} | HELD-OUT TEST ===')
    report_text = classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        digits=4,
        zero_division=0,
    )
    print(report_text)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report_dict).transpose().to_csv(
        os.path.join(output_dir, 'test_classification_report.csv')
    )

    cm = confusion_matrix(y_true, y_pred, labels=VALID_LABELS)
    pd.DataFrame(cm, index=VALID_LABELS, columns=VALID_LABELS).to_csv(
        os.path.join(output_dir, 'test_confusion_matrix.csv')
    )

    pred_rows = []
    for (cid, ti), true_label, pred_label in zip(data['test_meta'], y_true, y_pred):
        pred_rows.append({
            'conversation_id': cid,
            'turn_index': ti,
            'true_label': true_label,
            'predicted_label': pred_label,
        })
    pd.DataFrame(pred_rows).to_csv(os.path.join(output_dir, 'test_predictions.csv'), index=False)

    macro_f1 = float(report_dict['macro avg']['f1-score'])
    weighted_f1 = float(report_dict['weighted avg']['f1-score'])
    accuracy = float(report_dict['accuracy'])

    result = {
        'model': model_label,
        'cv': cv_run,
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'test_examples': len(y_true),
    }
    for label in VALID_LABELS:
        result[f'{label.lower()}_f1'] = float(report_dict[label]['f1-score'])
        result[f'{label.lower()}_support'] = int(report_dict[label]['support'])
    return result

In [18]:
# Cell 18 — Train one model on one frozen CV run

def train_one_model_one_cv(model_label, model_name, cv_run):
    print('\n' + '#' * 90)
    print(f'{model_label} | CV{cv_run}')
    print('#' * 90)

    set_seed(RANDOM_STATE)
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)

    # Repeat leakage audit immediately before this training run.
    audit_cv_fold(cv_run, verbose=True)
    data = build_one_cv_data(cv_run)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    train_ds, val_ds, test_ds, y_train_ids, y_val_ids, y_test_ids = make_hf_datasets(tokenizer, data)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Balanced class weights from FINAL TRAIN only.
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(len(VALID_LABELS)),
        y=y_train_ids,
    )
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
    print('TRAIN-only class weights:', dict(zip(VALID_LABELS, class_weights)))

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(VALID_LABELS),
        id2label=id2label,
        label2id=label2id,
    )

    safe_model = re.sub(r'[^A-Za-z0-9_.-]+', '_', model_label)
    output_dir = os.path.join(RESULTS_DIR, safe_model, f'cv{cv_run}')
    os.makedirs(output_dir, exist_ok=True)

    best_cb = BestValidationStateCallback(metric_name='eval_macro_f1')

    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy='epoch',
        save_strategy='no',
        logging_strategy='epoch',
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
        load_best_model_at_end=False,
        report_to='none',
        fp16=torch.cuda.is_available(),
    )

    trainer = WeightedTrainer(
        class_weights=class_weights_tensor,
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[best_cb],
    )

    trainer.train()

    assert best_cb.best_state_dict is not None, 'No best validation model state captured.'
    trainer.model.load_state_dict(best_cb.best_state_dict, strict=True)

    print('\nBest validation epoch:', best_cb.best_epoch)
    print('Best validation Macro-F1:', best_cb.best_metric)

    best_model_dir = os.path.join(output_dir, 'best_validation_model')
    os.makedirs(best_model_dir, exist_ok=True)
    trainer.model.save_pretrained(best_model_dir)
    tokenizer.save_pretrained(best_model_dir)

    result = evaluate_test(
        trainer, test_ds, y_test_ids, data, model_label, cv_run, output_dir
    )
    result['best_val_macro_f1'] = best_cb.best_metric
    result['best_val_epoch'] = best_cb.best_epoch
    result['learning_rate'] = LEARNING_RATE
    result['weight_decay'] = WEIGHT_DECAY
    result['seed'] = RANDOM_STATE
    result['final_train_examples'] = len(data['X_train'])
    result['val_examples'] = len(data['X_val'])

    pd.DataFrame([result]).to_csv(os.path.join(output_dir, 'test_metrics.csv'), index=False)

    del trainer, model, tokenizer, train_ds, val_ds, test_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## Run configuration

RoBERTa and MiniLM are evaluated on all five frozen conversation-level folds using the settings defined above.


In [19]:
# Cell 19 — Execute enabled models and requested folds
all_results = []

for model_label, cfg in MODEL_CONFIGS.items():
    if not cfg['enabled']:
        print(f'SKIP {model_label} (disabled)')
        continue

    for cv_run in RUN_FOLDS:
        result = train_one_model_one_cv(
            model_label=model_label,
            model_name=cfg['model_name'],
            cv_run=cv_run,
        )
        all_results.append(result)

        # Save after every completed run.
        pd.DataFrame(all_results).to_csv(
            os.path.join(RESULTS_DIR, 'running_model_cv_results.csv'),
            index=False,
        )

results_df = pd.DataFrame(all_results)
display(results_df)

SKIP BERT-base (disabled)
SKIP DistilBERT (disabled)

##########################################################################################
RoBERTa | CV1
##########################################################################################
CV1: PASS — no conversation-ID or target-turn leakage; LLM provenance verified.
  Train/Val/Test conversations: 528 132 165
  Original target rows: 2035 489 655
  LLM rows / source conversations: 200 66


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Chitchat F1,Interpret F1,Reveal F1,Revise F1
1,0.847400,0.783682,0.804183,0.635940,0.581527,0.577798,0.812826,0.535714,0.868381,0.795987,0.111111
2,0.623800,0.795779,0.828897,0.657304,0.552848,0.579720,0.825786,0.528302,0.882006,0.813333,0.095238
3,0.503600,0.824589,0.840304,0.670991,0.601567,0.615048,0.837115,0.545455,0.891207,0.823529,0.200000
4,0.400200,0.889839,0.821293,0.612271,0.554806,0.566361,0.828761,0.491228,0.884793,0.826923,0.062500
5,0.327700,0.897568,0.830798,0.621712,0.559466,0.574405,0.831636,0.500000,0.888889,0.825397,0.083333



Best validation epoch: 3.0
Best validation Macro-F1: 0.6150477776803718



=== MiniLM | CV5 | HELD-OUT TEST ===
              precision    recall  f1-score   support

    Chitchat     0.8605    0.6167    0.7184        60
   Interpret     0.8341    0.9526    0.8895       359
      Reveal     0.9298    0.7871    0.8525       202
      Revise     0.2222    0.1667    0.1905        12

    accuracy                         0.8531       633
   macro avg     0.7117    0.6308    0.6627       633
weighted avg     0.8556    0.8531    0.8482       633



,model,cv,accuracy,macro_f1,weighted_f1,test_examples,chitchat_f1,chitchat_support,interpret_f1,interpret_support,reveal_f1,reveal_support,revise_f1,revise_support,best_val_macro_f1,best_val_epoch,learning_rate,weight_decay,seed,final_train_examples,val_examples
0,RoBERTa,1,0.789313,0.589271,0.787316,655,0.472222,53,0.845938,377,0.800830,210,0.238095,15,0.607964,2.0,0.00002,0.01,42,5290,489
1,RoBERTa,2,0.836276,0.600918,0.832206,623,0.683544,50,0.882431,358,0.837696,201,0.000000,14,0.618840,2.0,0.00002,0.01,42,5308,503
2,RoBERTa,3,0.818612,0.582702,0.817363,634,0.637363,52,0.877524,351,0.815920,217,0.000000,14,0.567185,4.0,0.00002,0.01,42,5296,504
3,RoBERTa,4,0.829653,0.644162,0.834564,634,0.600000,53,0.881780,368,0.844869,201,0.250000,12,0.567365,3.0,0.00002,0.01,42,5293,507
4,RoBERTa,5,0.846761,0.632874,0.839848,633,0.685714,60,0.883721,359,0.850949,202,0.111111,12,0.566775,3.0,0.00002,0.01,42,5275,526
5,MiniLM,1,0.836641,0.621852,0.824458,655,0.485714,53,0.884521,377,0.841310,210,0.275862,15,0.610614,3.0,0.00002,0.01,42,5290,489
6,MiniLM,2,0.833066,0.602302,0.833778,623,0.683544,50,0.881579,358,0.844086,201,0.000000,14,0.594369,2.0,0.00002,0.01,42,5308,503
7,MiniLM,3,0.788644,0.589045,0.796214,634,0.674419,52,0.846462,351,0.792746,217,0.042553,14,0.582437,2.0,0.00002,0.01,42,5296,504
8,MiniLM,4,0.834385,0.639499,0.833085,634,0.600000,53,0.881971,368,0.840731,201,0.235294,12,0.597293,3.0,0.00002,0.01,42,5293,507
9,MiniLM,5,0.853081,0.662734,0.848224,633,0.718447,60,0.889467,359,0.852547,202,0.190476,12,0.615048,3.0,0.00002,0.01,42,5275,526


In [20]:
# Cell 20 — Completed-run summary (mean ± SD when multiple folds are available)
if len(results_df) == 0:
    print('No model runs completed.')
else:
    metrics = [
        'accuracy', 'macro_f1', 'weighted_f1',
        'chitchat_f1', 'interpret_f1', 'reveal_f1', 'revise_f1'
    ]

    summary_rows = []
    for model_name, group in results_df.groupby('model'):
        row = {'Model': model_name, 'Completed_folds': len(group)}
        for m in metrics:
            mean = group[m].mean()
            std = group[m].std(ddof=1) if len(group) > 1 else np.nan
            row[f'{m}_mean'] = mean
            row[f'{m}_std'] = std
            row[f'{m}_mean±sd'] = f'{mean:.4f} ± {std:.4f}' if len(group) > 1 else f'{mean:.4f} (1 fold)'
        summary_rows.append(row)

    model_cv_summary = pd.DataFrame(summary_rows)
    display(model_cv_summary)
    model_cv_summary.to_csv(
        os.path.join(RESULTS_DIR, 'model_5cv_summary.csv'),
        index=False,
    )

,Model,Completed_folds,accuracy_mean,accuracy_std,accuracy_mean±sd,macro_f1_mean,macro_f1_std,macro_f1_mean±sd,weighted_f1_mean,weighted_f1_std,weighted_f1_mean±sd,chitchat_f1_mean,chitchat_f1_std,chitchat_f1_mean±sd,interpret_f1_mean,interpret_f1_std,interpret_f1_mean±sd,reveal_f1_mean,reveal_f1_std,reveal_f1_mean±sd,revise_f1_mean,revise_f1_std,revise_f1_mean±sd
0,MiniLM,5,0.829163,0.024043,0.8292 ± 0.0240,0.623086,0.029299,0.6231 ± 0.0293,0.827152,0.019284,0.8272 ± 0.0193,0.632425,0.092671,0.6324 ± 0.0927,0.876800,0.017248,0.8768 ± 0.0172,0.834284,0.023696,0.8343 ± 0.0237,0.148837,0.121236,0.1488 ± 0.1212
1,RoBERTa,5,0.824123,0.021982,0.8241 ± 0.0220,0.609985,0.027146,0.6100 ± 0.0271,0.822260,0.021241,0.8223 ± 0.0212,0.615769,0.087759,0.6158 ± 0.0878,0.874279,0.016012,0.8743 ± 0.0160,0.830053,0.021025,0.8301 ± 0.0210,0.119841,0.122194,0.1198 ± 0.1222
